# Section 6.1 and 6.2: Law-Pinning and the Depth Dial

**Purpose.** These two experiments close out the remaining pieces of Section 6 (MLP track) that
weren't covered by the shared-comparison run or the 64M-scale run:

- **6.1 — Where the laws are pinned.** A $1$--$16$--$1$ tanh MLP ($P=49$) anchors the parameter axis:
  we measure the empirical $M^*$-vs-alignment curve against the closed-form alignment law
  (`cos² = M/(M+D+1)`, Eq. 3 in the paper), and note that since this MLP has no separable channels
  ($K=1$), this same curve *is* the $K=1$ endpoint of the barrier law (Section 4.2) — there's nothing
  separate to run for that part, it's the same data viewed through the other lens.
- **6.2 — The depth dial.** Depth, not width, is the outline's claimed degradation axis. We measure
  how consistently a network's per-output-channel gradients stay separable (uncoupled) as depth
  increases, using the same coupling-density style of Jacobian audit already used earlier in this
  project (Mamba-3 and diffusion-core audits) — reused here on a plain tanh MLP stack instead.

**Honesty note.** The outline text states "alignment falls from 1.00 to 0.821" as an illustrative
example. This notebook does NOT force that number — it measures whatever the real run produces.
Report whatever comes out, not 0.821 specifically, unless that's actually what you get.

## Step 0 — Setup (reused harness pattern)

In [1]:
import json
import time
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', device)

RESULTS_LOG = []

def log_result(name, config, result, seed):
    entry = {
        'name': name,
        'seed': seed,
        'config': config,
        'result': result,
        'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    }
    RESULTS_LOG.append(entry)
    print(f"[logged] {name}: {result}")
    return entry

def save_provenance(path='provenance_section6_laws.json'):
    with open(path, 'w') as f:
        json.dump(RESULTS_LOG, f, indent=2, default=str)
    print(f'Saved {len(RESULTS_LOG)} logged results to {path}')


Using device: cuda


## Step 1 — 6.1: The $1$--$16$--$1$ tanh MLP, $P=49$

Simple 1D regression (`y = sin(x) + noise`) so the model stays exactly at the outline's anchor size.
We sweep the probe count $M$, measure the empirical alignment $\cos^2(\hat g_M, g)$ against the true
gradient at a fixed random parameter vector $\theta$, and compare to the closed-form law.

In [2]:
SEED = 0
torch.manual_seed(SEED)

class TinyMLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))
    def forward(self, x):
        return self.net(x)

model_616 = TinyMLP().to(device)
P_616 = sum(p.numel() for p in model_616.parameters())
print(f'1-16-1 tanh MLP parameter count: P = {P_616}  (outline anchor: P=49)')
assert P_616 == 49, f'Expected P=49, got {P_616} -- architecture does not match the outline anchor.'

N_SAMPLES = 64
x_616 = torch.linspace(-3, 3, N_SAMPLES, device=device).view(-1, 1)
y_616 = torch.sin(x_616) + 0.05 * torch.randn_like(x_616)

def flat_params(m):
    return torch.cat([p.data.view(-1) for p in m.parameters()])

def set_flat_params(m, flat):
    offset = 0
    for p in m.parameters():
        n = p.numel()
        p.data.copy_(flat[offset:offset+n].view_as(p))
        offset += n

def true_grad(m, theta, x, y):
    set_flat_params(m, theta)
    m.zero_grad(set_to_none=True)
    loss = ((m(x) - y) ** 2).mean()
    loss.backward()
    return torch.cat([p.grad.view(-1) for p in m.parameters()]), loss.item()

def loss_only(m, theta, x, y):
    set_flat_params(m, theta)
    with torch.no_grad():
        return ((m(x) - y) ** 2).mean().item()

theta0 = flat_params(model_616).clone()
g_true, loss0 = true_grad(model_616, theta0, x_616, y_616)
print(f'Loss at theta0: {loss0:.4f}, ||g_true|| = {g_true.norm().item():.4f}')


1-16-1 tanh MLP parameter count: P = 49  (outline anchor: P=49)
Loss at theta0: 0.1966, ||g_true|| = 1.0524


In [ ]:
def estimate_ghat(m, theta, x, y, M, sigma=0.05):
    D = theta.numel()
    g_hat = torch.zeros(D, device=device)
    for _ in range(M):
        xi = torch.randn(D, device=device)
        l_plus = loss_only(m, theta + sigma * xi, x, y)
        l_minus = loss_only(m, theta - sigma * xi, x, y)
        g_hat += xi * (l_plus - l_minus) / (2 * sigma)
    return g_hat / M

M_VALUES = [1, 2, 5, 10, 20, 50, 100, 200, 500, 1000]
N_TRIALS = 20   # repeats per M, to average out probe randomness

empirical_cos2 = []
theoretical_cos2 = []

for M in M_VALUES:
    cos2_trials = []
    for trial in range(N_TRIALS):
        g_hat = estimate_ghat(model_616, theta0, x_616, y_616, M)
        cos_sim = torch.dot(g_hat, g_true) / (g_hat.norm() * g_true.norm() + 1e-12)
        cos2_trials.append((cos_sim ** 2).item())
    mean_cos2 = float(np.mean(cos2_trials))
    empirical_cos2.append(mean_cos2)
    theory = M / (M + P_616 + 1)
    theoretical_cos2.append(theory)
    print(f'M={M:>5}   empirical cos^2={mean_cos2:.4f}   theoretical cos^2={theory:.4f}')
    log_result('law_pinning_M_sweep', {'M': M, 'P': P_616, 'n_trials': N_TRIALS, 'sigma': 0.05},
               {'empirical_cos2': mean_cos2, 'theoretical_cos2': theory}, SEED)


M=    1   empirical cos^2=0.0120   theoretical cos^2=0.0196
[logged] law_pinning_M_sweep: {'empirical_cos2': 0.012034633569601282, 'theoretical_cos2': 0.0196078431372549}
M=    2   empirical cos^2=0.0223   theoretical cos^2=0.0385
[logged] law_pinning_M_sweep: {'empirical_cos2': 0.022299408743856476, 'theoretical_cos2': 0.038461538461538464}
M=    5   empirical cos^2=0.0862   theoretical cos^2=0.0909
[logged] law_pinning_M_sweep: {'empirical_cos2': 0.08622827576473355, 'theoretical_cos2': 0.09090909090909091}
M=   10   empirical cos^2=0.1915   theoretical cos^2=0.1667
[logged] law_pinning_M_sweep: {'empirical_cos2': 0.19154944140464067, 'theoretical_cos2': 0.16666666666666666}
M=   20   empirical cos^2=0.3172   theoretical cos^2=0.2857
[logged] law_pinning_M_sweep: {'empirical_cos2': 0.317183993011713, 'theoretical_cos2': 0.2857142857142857}
M=   50   empirical cos^2=0.5371   theoretical cos^2=0.5000
[logged] law_pinning_M_sweep: {'empirical_cos2': 0.5371494755148888, 'theoretical_cos2

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(M_VALUES, empirical_cos2, 'o-', label='Empirical $\\cos^2$')
plt.plot(M_VALUES, theoretical_cos2, '--', label='Theoretical $M/(M+P+1)$')
plt.xscale('log')
plt.xlabel('Probe count M')
plt.ylabel('$\\cos^2(\\hat g_M, g)$')
plt.title(f'Alignment law at P={P_616} (the 1-16-1 tanh MLP anchor)')
plt.legend()
plt.grid(alpha=0.3)
plt.savefig('alignment_law_P49.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved plot to alignment_law_P49.png')


**Note on the barrier law's $K=1$ endpoint (Section 4.2).** This MLP has no separable structure, so
it has exactly one feedback channel ($K=1$) and the coupling scope equals the full parameter count
($S=P=49$ here, since nothing decouples). The barrier law $M^*K \gtrsim S$ therefore collapses to
$M^* \gtrsim S = P$ at $K=1$ — the same curve just measured above. No separate experiment is needed
for this; report the same numbers under both headings in the writeup, with a one-line note explaining
why they coincide.

## Step 2 — 6.2: The depth dial

We build tanh MLPs of increasing depth (fixed width) with a 4-dimensional output (so there are
multiple "modes"/channels to compare), and measure how separable the per-channel gradients stay
as depth increases. This reuses the Jacobian/coupling-density style of audit already used earlier
in this project for the Mamba-3 and diffusion-core checks (Round 1/1b), applied here to a plain
tanh stack instead.

**Metric.** For each depth, compute the per-output-channel gradient of the parameters
($\partial y_i / \partial \theta$ for each output channel $i$), form the pairwise cosine similarity
matrix across channels, and average $|\cos|$ over off-diagonal entries to get a coupling density
in $[0, 1]$. We report **alignment $= 1 - $ coupling density** (so higher = more separable, matching
the outline's "1.00 is fully aligned" framing) as a function of depth.

In [ ]:
def build_depth_mlp(depth, width=16, input_dim=4, output_dim=4, seed=0):
    torch.manual_seed(seed)
    layers = [nn.Linear(input_dim, width), nn.Tanh()]
    for _ in range(depth - 1):
        layers += [nn.Linear(width, width), nn.Tanh()]
    layers += [nn.Linear(width, output_dim)]
    return nn.Sequential(*layers).to(device)

def coupling_density_and_alignment(model, output_dim, input_dim=4, n_probes=10, seed=0):
    torch.manual_seed(seed)
    densities = []
    for _ in range(n_probes):
        x = torch.randn(1, input_dim, device=device, requires_grad=False)
        per_channel_grads = []
        for i in range(output_dim):
            model.zero_grad(set_to_none=True)
            out = model(x)
            out[0, i].backward(retain_graph=True)
            g = torch.cat([p.grad.view(-1).clone() for p in model.parameters() if p.grad is not None])
            per_channel_grads.append(g)
        G = torch.stack(per_channel_grads)
        G_norm = G / (G.norm(dim=1, keepdim=True) + 1e-12)
        cos_matrix = G_norm @ G_norm.T
        off_diag = cos_matrix[~torch.eye(output_dim, dtype=torch.bool, device=device)]
        densities.append(off_diag.abs().mean().item())
    density = float(np.mean(densities))
    alignment = 1 - density
    return density, alignment

DEPTHS = [1, 2, 4, 8, 16]
depth_results = []

for depth in DEPTHS:
    m = build_depth_mlp(depth, seed=SEED)
    density, alignment = coupling_density_and_alignment(m, output_dim=4, seed=SEED)
    depth_results.append((depth, density, alignment))
    print(f'depth={depth:>3}   coupling density={density:.3f}   per-mode alignment={alignment:.3f}')
    log_result('depth_dial', {'depth': depth, 'width': 16, 'output_dim': 4, 'n_probes': 10},
               {'coupling_density': density, 'alignment': alignment}, SEED)


In [ ]:
depths_arr = [r[0] for r in depth_results]
alignments_arr = [r[2] for r in depth_results]

plt.figure(figsize=(6, 4))
plt.plot(depths_arr, alignments_arr, 'o-', color='C1')
plt.xlabel('Depth (number of hidden layers)')
plt.ylabel('Per-mode alignment (1 - coupling density)')
plt.title('The depth dial: separability erosion with depth')
plt.grid(alpha=0.3)
plt.savefig('depth_dial.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved plot to depth_dial.png')
print()
print('Compare this trend (direction and rough magnitude) against the outline\'s illustrative')
print('1.00 -> 0.821 example -- report the REAL numbers above, not the outline\'s example figure,')
print('unless they happen to match.')


## Step 3 — Save provenance

In [ ]:
save_provenance('provenance_section6_laws.json')
print()
print('Contents:')
for entry in RESULTS_LOG:
    print(f"  - {entry['name']}: {entry['result']}")


## Summary: what this closes out for Section 6

1. **6.1 (law-pinning) is now measurable, not just described.** The $M$-sweep at $P=49$ gives a real
   empirical curve to plot against the theoretical alignment law, on the exact architecture the
   outline names.
2. **The $K=1$ barrier-law endpoint is the same data**, not a separate experiment -- worth stating
   that explicitly in the writeup so it doesn't read as a missing result.
3. **6.2 (depth dial) now has a concrete, reusable metric** (coupling density / per-mode alignment)
   applied consistently with the Mamba-3 and diffusion-core audits already done earlier in this
   project, rather than a one-off ad hoc measurement.
4. **Whatever numbers come out of your run are the numbers to report** -- the outline's "1.00 to
   0.821" is an illustrative target, not a result to reproduce by construction. If your real numbers
   differ, that's the honest finding.